# Breast Cancer Classification — Model Training & Evaluation

**Dataset:** Breast Cancer Wisconsin (Diagnostic), UCI ML Repository (via `sklearn.datasets`)
569 instances, 30 numeric features, binary target (0 = malignant, 1 = benign).

Run all cells top to bottom. This is the notebook version of `train_models.py`,
meant to be executed on BITS Virtual Lab for the assignment screenshot.

In [ ]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

## 1. Load the dataset

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print(f"Shape: {X.shape[0]} instances x {X.shape[1]} features")
print(f"Target names: {list(data.target_names)}  (0={data.target_names[0]}, 1={data.target_names[1]})")
y.value_counts()

## 2. Train/test split and feature scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the raw (unscaled) test split — this becomes test_data.csv for the app
test_df = X_test.copy()
test_df["target"] = y_test.values
test_df.to_csv("../test_data.csv", index=False)
print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

## 3. Train and evaluate all 5 models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
}

file_keys = {
    "Logistic Regression": "logistic_regression",
    "Decision Tree": "decision_tree",
    "kNN": "knn",
    "Naive Bayes": "naive_bayes",
    "Random Forest (Ensemble)": "random_forest",
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    proba = model.predict_proba(X_test_scaled)[:, 1]

    metrics = {
        "Accuracy": accuracy_score(y_test, preds),
        "AUC": roc_auc_score(y_test, proba),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds),
        "MCC": matthews_corrcoef(y_test, preds),
    }
    results[name] = metrics

    print(f"--- {name} ---")
    print(pd.Series(metrics).round(4).to_string())
    print(f"Confusion matrix:\n{confusion_matrix(y_test, preds)}\n")

    joblib.dump(model, f"{file_keys[name]}.pkl")

joblib.dump(scaler, "scaler.pkl")

## 4. Comparison table

In [ ]:
results_df = pd.DataFrame(results).T
results_df.index.name = "ML Model Name"
results_df = results_df.round(4)
results_df.to_csv("metrics.csv")
results_df